# Bilinear convolutional model analysis

<i>Extends the work of https://github.com/tdooms/bilinear-decomposition</i>

### Set up

In [1]:
%load_ext autoreload
%autoreload 2

import torch
import plotly.express as px

from image import CNNModel, MNIST, Model
from einops import einsum
from kornia.augmentation import RandomGaussianNoise
from image import plot_eigenspectrum, plot_explanation

device = "cuda:0"
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)

#### Test samples corresponding with digit 5

In [ ]:
test_indices_digit_5 = (test.y == 5).nonzero(as_tuple=True)[0]
test_digit_5 = test.x[test_indices_digit_5]

## Bilinear convolutional model hyperparameter sweep

In [2]:
from itertools import product
import pandas as pd


def sweep_kernel(
    epochs_list,
    d_hidden_list,
    wd_list,
    noise_std_list,
    kernel_size_list,
    *,
    device=device,
    seed=42,
    padding=0,
    lr=None,
    batch_size=None,
):
    results = []
    best_model = None
    best_val_acc = float("-inf")

    for epochs, d_hidden, wd, noise_std, kernel_size in product(
        epochs_list, d_hidden_list, wd_list, noise_std_list, kernel_size_list
    ):
        config = {
            "epochs": epochs,
            "d_hidden": d_hidden,
            "wd": wd,
            "padding": padding,
            "kernel_size": kernel_size,
            "seed": seed,
        }
        if lr is not None:
            config["lr"] = lr
        if batch_size is not None:
            config["batch_size"] = batch_size

        model = CNNModel.from_config(**config).to(device)
        transform = RandomGaussianNoise(std=noise_std)
        metrics = model.fit(train, test, transform)

        final_train_acc = float(metrics["train/acc"].iloc[-1])
        final_val_acc = float(metrics["val/acc"].iloc[-1])

        results.append(
            {
                "epochs": epochs,
                "d_hidden": d_hidden,
                "wd": wd,
                "noise_std": noise_std,
                "kernel_size": kernel_size,
                "final_train_acc": final_train_acc,
                "final_val_acc": final_val_acc,
            }
        )

        if final_val_acc > best_val_acc:
            best_val_acc = final_val_acc
            best_model = model

    results_df = pd.DataFrame(results).sort_values(
        ["final_val_acc", "final_train_acc"], ascending=[False, False]
    )
    return results_df, best_model


results_df, best_model = sweep_kernel(
    epochs_list=[5, 10, 20],
    d_hidden_list=[64, 128, 256],
    wd_list=[0, 0.1, 0.5, 1.0],
    noise_std_list=[0, 0.25, 0.5],
    kernel_size_list=[3, 5, 7, 14, 28],
)
results_df.to_csv("sweep_kernel_results.csv", index=False)
results_df.head()

train/loss: 0.117, train/acc: 0.970, val/loss: 0.122, val/acc: 0.968: 100%|██████████| 5/5 [00:03<00:00,  1.66it/s]
train/loss: 0.075, train/acc: 0.980, val/loss: 0.078, val/acc: 0.978: 100%|██████████| 5/5 [00:02<00:00,  2.42it/s]
train/loss: 0.068, train/acc: 0.981, val/loss: 0.068, val/acc: 0.980: 100%|██████████| 5/5 [00:01<00:00,  2.58it/s]
train/loss: 0.089, train/acc: 0.976, val/loss: 0.083, val/acc: 0.978: 100%|██████████| 5/5 [00:01<00:00,  4.05it/s]
train/loss: 0.282, train/acc: 0.927, val/loss: 0.278, val/acc: 0.929: 100%|██████████| 5/5 [00:01<00:00,  4.65it/s]
train/loss: 0.133, train/acc: 0.965, val/loss: 0.125, val/acc: 0.968: 100%|██████████| 5/5 [00:02<00:00,  2.08it/s]
train/loss: 0.085, train/acc: 0.977, val/loss: 0.079, val/acc: 0.978: 100%|██████████| 5/5 [00:02<00:00,  2.43it/s]
train/loss: 0.078, train/acc: 0.978, val/loss: 0.069, val/acc: 0.980: 100%|██████████| 5/5 [00:01<00:00,  2.64it/s]
train/loss: 0.099, train/acc: 0.973, val/loss: 0.083, val/acc: 0.979: 10

,epochs,d_hidden,wd,noise_std,kernel_size,final_train_acc,final_val_acc
443,20,128,0.1,0.25,14,0.992945,0.9870
517,20,256,0.5,0.25,7,0.994966,0.9869
533,20,256,1.0,0.25,14,0.994208,0.9869
457,20,128,0.5,0.25,7,0.993130,0.9869
493,20,256,0.0,0.50,14,0.981008,0.9869


# Bilinear MLP

Config is
- Epochs = 40
- hidden dimension = 512
- Learning rate = 0.001
- AdamW optimizer
- Cosine scheduler
- weight decay = 1
- std Gaussian noise = 0.5

In [18]:
model = Model.from_config(epochs=40, d_hidden=512, wd=1).to(device)
metrics = model.fit(train, test, RandomGaussianNoise(std=0.5))

train/loss: 0.106, train/acc: 0.968, val/loss: 0.076, val/acc: 0.980: 100%|██████████| 40/40 [00:08<00:00,  4.88it/s]


The model is trained, we can now analyse the eigenspectrum for each output class of the trained model

In [19]:
for digit in range(10):
    fig = plot_eigenspectrum(model, eigenvectors=4, digit=digit)
    fig.write_image(f"linear_{digit}.png", scale=2)

With the same technique, we can also do instance-based explainability and understand how the model reached its conclusion and this solely based on the weights of the model.

In [120]:
fig = plot_explanation(model, test_digit_5[10])
fig.write_image("prediction_explanation_linear_model.png", scale=2)
fig

# Bilinear CNNs

### Kernel size variation

#### Kernel size = 3

In [2]:
# Use the CNN bilinear model
model_kernel_size_3 = CNNModel.from_config(epochs=5, d_hidden=256, padding=0, kernel_size=3, wd=1).to(device)
_ = model_kernel_size_3.fit(train, test, RandomGaussianNoise(std=0.25))

train/loss: 0.087, train/acc: 0.976, val/loss: 0.084, val/acc: 0.976: 100%|██████████| 5/5 [00:11<00:00,  2.29s/it]


In [9]:
for digit in range(10):
    fig = plot_eigenspectrum(model_kernel_size_3, eigenvectors=4, digit=digit)
    fig.write_image(f"conv_3_{digit}.png", scale=2)
    

#### Kernel size = 5

In [2]:
# Use the CNN bilinear model
model_kernel_size_5 = CNNModel.from_config(epochs=5, d_hidden=256, padding=0, kernel_size=5, wd=1).to(device)
_ = model_kernel_size_5.fit(train, test, RandomGaussianNoise(std=0.25))

train/loss: 0.055, train/acc: 0.985, val/loss: 0.056, val/acc: 0.982: 100%|██████████| 5/5 [00:07<00:00,  1.42s/it]


In [3]:
for digit in range(10):
    fig = plot_eigenspectrum(model_kernel_size_5, eigenvectors=4, digit=digit)
    fig.write_image(f"conv_5_{digit}.png", scale=2)

In [62]:
plot_eigenspectrum(model_kernel_size_5, eigenvectors=5, digit=9)

#### Kernel size = 7

In [12]:
model_kernel_size_7 = CNNModel.from_config(epochs=5, d_hidden=256, padding=0, kernel_size=7, wd=1).to(device)
_ = model_kernel_size_7.fit(train, test, RandomGaussianNoise(std=0.25))

train/loss: 0.050, train/acc: 0.986, val/loss: 0.052, val/acc: 0.984: 100%|██████████| 5/5 [00:05<00:00,  1.14s/it]


In [13]:
for digit in range(10):
    fig = plot_eigenspectrum(model_kernel_size_7, eigenvectors=4, digit=digit)
    fig.write_image(f"conv_7_{digit}.png", scale=2)

In [88]:
plot_eigenspectrum(model_kernel_size_7, eigenvectors=5, digit=9)

## Kernel size = 14

In [14]:
model_kernel_size_14 = CNNModel.from_config(epochs=5, d_hidden=256, padding=0, kernel_size=14, wd=1).to(device)
_ = model_kernel_size_14.fit(train, test, RandomGaussianNoise(std=0.25))

train/loss: 0.059, train/acc: 0.984, val/loss: 0.058, val/acc: 0.984: 100%|██████████| 5/5 [00:03<00:00,  1.33it/s]


In [15]:
for digit in range(10):
    fig = plot_eigenspectrum(model_kernel_size_14, eigenvectors=4, digit=digit)
    fig.write_image(f"conv_14_{digit}.png", scale=2)

In [90]:
plot_eigenspectrum(model_kernel_size_14, eigenvectors=5, digit=9)

#### Kernel size = 28

Equivalent to a linear layer

In [16]:
model_kernel_size_28 = CNNModel.from_config(epochs=5, d_hidden=256, padding=0, kernel_size=28, wd=1).to(device)
_ = model_kernel_size_28.fit(train, test, RandomGaussianNoise(std=0.25))

train/loss: 0.262, train/acc: 0.931, val/loss: 0.231, val/acc: 0.941: 100%|██████████| 5/5 [00:01<00:00,  4.97it/s]


In [17]:
for digit in range(10):
    fig = plot_eigenspectrum(model_kernel_size_28, eigenvectors=4, digit=digit)
    fig.write_image(f"conv_28_{digit}.png", scale=2)

In [72]:
plot_eigenspectrum(model_kernel_size_28, eigenvectors=5, digit=9)

### Instance-based explainability for kernel size K = 14

In [3]:
model = CNNModel.from_config(epochs=5, d_hidden=256, padding=0, kernel_size=14, wd=1).to(device)
_ = model.fit(train, test, RandomGaussianNoise(std=0.25))

train/loss: 0.059, train/acc: 0.984, val/loss: 0.058, val/acc: 0.984: 100%|██████████| 5/5 [00:04<00:00,  1.19it/s]


In [117]:
fig = plot_explanation(model, test_digit_5[10])
fig.write_image("prediction_explanation_conv_model.png", scale=2)

In [118]:
fig

## Plotting code

This code is used to combine all eigenspectrum analysis of each model for each digit.

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.font_manager as fm

# ── configuration ─────────────────────────────────────────────────────────────

INPUT_DIR  = Path(".")
OUTPUT_DIR = Path(".")

DIGITS = list(range(10))

ROW_LABELS = [
    "Linear",
    "Convolutional (K = 3)",
    "Convolutional (K = 5)",
    "Convolutional (K = 7)",
    "Convolutional (K = 14)",
    "Convolutional (K = 28)",
]

FILE_PATTERNS = [
    "linear_{digit}.png",
    "conv_3_{digit}.png",
    "conv_5_{digit}.png",
    "conv_7_{digit}.png",
    "conv_14_{digit}.png",
    "conv_28_{digit}.png",
]

LABEL_WIDTH       = 80  
DIGIT_LABEL_WIDTH = 140 
FONT_SIZE         = 28
DIGIT_FONT_SIZE   = 36
BG_COLOR          = (255, 255, 255)

def load_bold_font(size):
    path = fm.findfont(fm.FontProperties(weight="bold"))
    print(f"Using font: {path}")
    return ImageFont.truetype(path, size)

def vertical_label(text, height, width, font):
    tmp  = Image.new("RGBA", (1, 1))
    bbox = ImageDraw.Draw(tmp).textbbox((0, 0), text, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    canvas = Image.new("RGBA", (height, width), (255, 255, 255, 255))
    ImageDraw.Draw(canvas).text(
        ((height - tw) // 2, (width - th) // 2),
        text, font=font, fill=(0, 0, 0, 255)
    )
    return canvas.rotate(90, expand=True)

font_small = load_bold_font(FONT_SIZE)
font_large = load_bold_font(DIGIT_FONT_SIZE)

for digit in DIGITS:
    images = [Image.open(INPUT_DIR / p.format(digit=digit)).convert("RGB")
              for p in FILE_PATTERNS]

    w      = images[0].width
    images = [img.resize((w, int(img.height * w / img.width)), Image.LANCZOS)
              if img.width != w else img for img in images]
    total_h = sum(img.height for img in images)

    out = Image.new("RGB", (LABEL_WIDTH + w, total_h), BG_COLOR)
    y = 0
    for img, label in zip(images, ROW_LABELS):
        out.paste(img, (LABEL_WIDTH, y))
        out.paste(vertical_label(label, img.height, LABEL_WIDTH, font_small), (0, y))
        y += img.height

    out.save(OUTPUT_DIR / f"eigenspectrum_{digit}.png")
    print(f"eigenspectrum_{digit}.png saved")

digit_imgs = [Image.open(OUTPUT_DIR / f"eigenspectrum_{d}.png").convert("RGB")
              for d in DIGITS]

total_h = sum(img.height for img in digit_imgs)
img_w   = digit_imgs[0].width
mega    = Image.new("RGB", (DIGIT_LABEL_WIDTH + img_w, total_h), BG_COLOR)

y = 0
for d, img in zip(DIGITS, digit_imgs):
    mega.paste(img, (DIGIT_LABEL_WIDTH, y))
    mega.paste(vertical_label(f"Digit {d}", img.height, DIGIT_LABEL_WIDTH, font_large), (0, y))
    y += img.height

mega.save(OUTPUT_DIR / "eigenspectrum_all.png")
print("eigenspectrum_all.png saved")

Using font: /rhea/scratch/brussel/116/vsc11642/CNN-Analysis-via-TN/bilinear-decomposition-main/.venv/lib/python3.14/site-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSans-Bold.ttf
Using font: /rhea/scratch/brussel/116/vsc11642/CNN-Analysis-via-TN/bilinear-decomposition-main/.venv/lib/python3.14/site-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSans-Bold.ttf
eigenspectrum_0.png saved
eigenspectrum_1.png saved
eigenspectrum_2.png saved
eigenspectrum_3.png saved
eigenspectrum_4.png saved
eigenspectrum_5.png saved
eigenspectrum_6.png saved
eigenspectrum_7.png saved
eigenspectrum_8.png saved
eigenspectrum_9.png saved
eigenspectrum_all.png saved
